# Pregunta 2 — Mapeo de inundaciones con Sentinel-1 y U-Net propia (Sen1Floods11 + Lima 2017)

Cuaderno único: entrenamiento, ablación, evaluación, **construcción de la región de Lima** y figuras.
Todo el código vive en `repo/src/`; el cuaderno solo lo orquesta. **Antes de empezar:** Entorno de ejecución → Cambiar tipo → **GPU T4**.

| Sección | Entregable |
|---|---|
| 4–5 | E1 U-Net propia · E2 ablación · E3 en distribución · E6 bono SAR+S2 |
| 6 | E4 generalización a Lima (Sentinel-1 17-mar-2017 + EMSN038) |
| 7 | Tablas (E2, E3, E4) |
| 8 | E5 fallos del radar (figuras) |

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Repositorio
Opción A: URL de GitHub del grupo. Opción B: subir `repo.zip` a `/content/` con el panel de archivos.

In [ ]:
import os, zipfile
REPO_URL = 'https://github.com/loupv99/sen1floods11-unet-lima'
if REPO_URL:
    !git clone -q $REPO_URL /content/repo
elif os.path.exists('/content/repo.zip'):
    zipfile.ZipFile('/content/repo.zip').extractall('/content')
assert os.path.exists('/content/repo/run_all.sh'), 'falta el repositorio: clonar o subir repo.zip'
%cd /content/repo

## 3. Dependencias (incluye las de Lima: rasterio, geopandas, Planetary Computer)

In [ ]:
!pip -q install -r requirements.txt

## 4. Configuración y prueba de humo (≈ 10 min, incluye la descarga de ~9 GB)
`SEEDS` se usa en TODAS las celdas siguientes: si se cambia, las corridas que falten se entrenan.
Con poco tiempo, `SEEDS = "0"` (una semilla; decirlo en el informe).

In [ ]:
SEEDS = "0 1 2"
USE_DRIVE = False   # True: guarda runs/ en Google Drive por si Colab se desconecta
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/p2_runs && rm -rf runs && ln -s /content/drive/MyDrive/p2_runs runs
!QUICK=1 bash run_all.sh 2>&1 | tail -30 && touch runs/.solo_prueba_de_humo

## 5. Experimento completo + Lima (E1, E2, E3, E4, E6) — varias horas en T4
Un solo comando: datos → Otsu → ablaciones (dorado vs. débiles, pérdidas, arquitectura) → bono S2 →
**construye Lima** (descarga EMSN038 ~2.3 GB y la escena Sentinel-1 del 17-mar-2017) → evalúa en Lima → fallos → tablas.
Es idempotente: si Colab se corta, volver a correr la celda retoma sin repetir lo terminado.

In [ ]:
!if [ -f runs/.solo_prueba_de_humo ]; then rm -rf runs/* runs/.solo_prueba_de_humo results; echo 'borrada la prueba de humo'; fi   # nunca borra corridas reales
!PERU=1 SEEDS="$SEEDS" bash run_all.sh 2>&1 | tee results_log.txt | grep -E "mejor IoU|\[(test|bolivia|valid|peru)\]|chips de Lima|polígonos"

## 6. Solo Lima (si el experimento ya se entrenó sin `PERU=1`)
No reentrena nada mientras `SEEDS` sea el mismo de la corrida: construye Lima, evalúa todos los modelos y Otsu, y rehace las tablas.

In [ ]:
!PERU=1 SEEDS="$SEEDS" bash run_all.sh 2>&1 | grep -E "\[peru\]|chips de Lima|polígonos|umbral Otsu"
from IPython.display import Image, display
if os.path.exists('results/lima_mapa.png'): display(Image('results/lima_mapa.png'))

## 7. Tablas (media ± desviación entre semillas, y brecha test → Bolivia / Lima)

In [ ]:
print(open('results/tablas.md', encoding='utf-8').read())

## 8. Figuras de fallos del radar (E5)
Paneles: VV, VH, (RGB S2 si hay), etiqueta, predicción y errores (rojo = FP, azul = FN).
La etiqueta del nombre del archivo es una **pista heurística**: confirmar cada caso mirando la imagen.

In [ ]:
import glob
BEST = 'A_dorado__s0'   # <- la mejor configuración en VALID
for split in ['test', 'bolivia', 'peru']:
    figs = sorted(glob.glob(f'runs/{BEST}/eval_{split}/fallos/*.png'))[:6]
    print(f'==== {split}: {len(figs)} figuras')
    for f in figs:
        print(os.path.basename(f)); display(Image(f))

## Límites de Lima que van en el informe
1. `Potential_Maximum_flood` (EMSN038, *Risk & Recovery*) es la extensión **máxima** de la crecida, no el agua del 17-mar.
2. Desfase: radar del 17-mar vs. WorldView-2 del 18 y 21-mar.
3. La escena viene como γ0 RTC (Planetary Computer) y se convierte a σ0 con el ángulo de incidencia real (43–46°): exacto en llano, aproximado en laderas.
4. El agua es ~1.25 % de los píxeles válidos: el IoU es muy sensible a pocos falsos positivos.